# NEWMA TCPD Oracle results

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display

START_DIR = Path.cwd().resolve()
NOTEBOOK_NAME = 'NEWMA_kerneldetector_TCPD_VISIBLE_TABLES.ipynb'
HERE = next(
    candidate for parent in (START_DIR, *START_DIR.parents)
    for candidate in (parent, parent / 'code')
    if (candidate / NOTEBOOK_NAME).exists()
)

code_candidates = [HERE, *HERE.parents]
code_candidates.extend(parent / 'OKAFF' for parent in HERE.parents)
CODE_ROOT = next(
    (
        candidate
        for candidate in code_candidates
        if (candidate / 'src' / 'kerneldetector.py').is_file()
    ),
    None,
)
if CODE_ROOT is None:
    raise FileNotFoundError('Could not locate the project src directory')

SRC_DIR = CODE_ROOT / 'src'
for module_dir in (HERE, SRC_DIR):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

import kerneldetector

if Path(kerneldetector.__file__).resolve().parent != SRC_DIR.resolve():
    raise RuntimeError('Restart the kernel so kerneldetector loads from src')

from kerneldetector import NEWMA, GaussianKernel
from onlinecp.algos import select_optimal_parameters
from onlinecp.utils import feature_functions as feat
from metrics import f_measure

DATASET_DIR = HERE / 'datasets'
ANNOTATIONS_FILE = HERE / 'annotations.json'
RESULTS = HERE / 'results' / 'NEWMA_kerneldetector_TCPD'
RESULTS.mkdir(parents=True, exist_ok=True)

REFERENCE_SIZE = 50
REFERENCE_THRESHOLD_ALPHA = 0.10
FIRST_ALARM_POSITION = 50
MATCH_MARGIN = 5
EVALUATION_START = FIRST_ALARM_POSITION - MATCH_MARGIN
SEED = 0

ORACLE_B_GRID = (25, 50, 100, 150, 250)
ORACLE_Q_GRID = (0.90, 0.95, 0.99, 0.995)
ORACLE_RHO_GRID = ('lambda2', 0.005, 0.01, 0.05, 0.10, 0.20)


## Oracle grid search and tables

In [2]:
EXCLUDED = {f'quality_control_{i}' for i in range(1, 6)} | {'uk_coal_employ'}

with ANNOTATIONS_FILE.open() as stream:
    annotations = json.load(stream)


def load_dataset(name):
    with (DATASET_DIR / f'{name}.json').open() as stream:
        metadata = json.load(stream)
    matrix = np.asarray([
        [np.nan if value is None else value for value in series['raw']]
        for series in metadata['series']
    ], dtype=float).T
    if not np.all(np.isfinite(matrix)):
        raise ValueError(f'{name} contains non-finite observations')
    return metadata, matrix


datasets = sorted(
    name for name in annotations
    if name not in EXCLUDED
    and json.loads((DATASET_DIR / f'{name}.json').read_text())['n_obs'] > REFERENCE_SIZE
)
metadata_by_name, matrices = {}, {}
for name in datasets:
    metadata_by_name[name], matrices[name] = load_dataset(name)


def retained_annotation(name):
    return {
        annotator: [point for point in points if point >= EVALUATION_START]
        for annotator, points in annotations[name].items()
    }


def parameters_from_B(B):
    lambda1, lambda2 = select_optimal_parameters(B)
    features = int((1 / 4) / (lambda1 + lambda2) ** 2)
    return float(lambda1), float(lambda2), features


def oracle_rhos_for_B(B):
    lambda2 = parameters_from_B(B)[1]
    return tuple(dict.fromkeys(
        lambda2 if rate == 'lambda2' else float(rate)
        for rate in ORACLE_RHO_GRID
    ))


def run_newma(name, dataset_number, B, q, rho):
    matrix = matrices[name]
    lambda1, lambda2, features = parameters_from_B(B)
    state = np.random.get_state()
    np.random.seed(SEED + dataset_number)
    gamma = float(GaussianKernel.est_gamma(matrix[:REFERENCE_SIZE]))
    frequencies = GaussianKernel(gamma=gamma).rff_sampler(matrix.shape[1])(features)
    np.random.set_state(state)
    detector = NEWMA(
        matrix[0],
        forget_factor=lambda1,
        forget_factor2=lambda2,
        feat_func=lambda x: feat.fourier_feat(x, frequencies),
        adapt_forget_factor=REFERENCE_THRESHOLD_ALPHA,
        thresholding_quantile=q,
    )
    for index, sample in enumerate(matrix):
        if index == REFERENCE_SIZE:
            detector.adapt_forget_factor = float(rho)
        detector.update(sample)
    statistic = np.asarray([item[0] for item in detector.stat_stored])
    threshold = np.asarray([item[1] for item in detector.stat_stored])
    return np.flatnonzero(
        (statistic > threshold)
        & (np.arange(len(statistic)) >= FIRST_ALARM_POSITION)
    ).tolist()


oracle_rows = []
for dataset_number, name in enumerate(datasets):
    data_type = (
        'multivariate' if metadata_by_name[name]['n_dim'] > 1 else 'univariate'
    )
    annotation = retained_annotation(name)
    print(f'[Oracle {dataset_number + 1:02d}/{len(datasets):02d}] {name}', flush=True)
    for B in ORACLE_B_GRID:
        for rho in oracle_rhos_for_B(B):
            for q in ORACLE_Q_GRID:
                alarms = run_newma(name, dataset_number, B, q, rho)
                oracle_rows.append({
                    'dataset': name,
                    'data type': data_type,
                    'B': B,
                    'rho': rho,
                    'q': q,
                    'F1': f_measure(annotation, alarms),
                })

oracle_grid = pd.DataFrame(oracle_rows)
oracle_f1 = oracle_grid.loc[
    oracle_grid.groupby('dataset')['F1'].idxmax()
].sort_values('dataset').reset_index(drop=True)

oracle_summary = (
    oracle_f1.groupby('data type', as_index=False)
    .agg(**{
        'Oracle F1': ('F1', 'mean'),
        'N datasets': ('dataset', 'nunique'),
    })
    [['data type', 'Oracle F1', 'N datasets']]
)
oracle_f1_table = oracle_f1[[
    'dataset', 'data type', 'B', 'rho', 'q', 'F1'
]].rename(columns={'F1': 'Oracle F1'})

oracle_summary.to_csv(RESULTS / 'newma_oracle_summary.csv', index=False)
oracle_f1_table.to_csv(
    RESULTS / 'newma_oracle_f1_by_dataset.csv', index=False
)

print('Overall Oracle results')
display(oracle_summary.round({'Oracle F1': 4}))

aggregate_excluded_datasets = {'gdp_iran', 'gdp_japan', 'ozone', 'robocalls'}
oracle_summary_excluding = (
    oracle_f1.loc[~oracle_f1['dataset'].isin(aggregate_excluded_datasets)]
    .groupby('data type', as_index=False)
    .agg(**{
        'Oracle F1': ('F1', 'mean'),
        'N datasets': ('dataset', 'nunique'),
    })
    [['data type', 'Oracle F1', 'N datasets']]
)
print(
    'Oracle results excluding gdp_iran, gdp_japan, ozone, and robocalls'
)
display(oracle_summary_excluding.round({'Oracle F1': 4}))

print('Oracle-F1: maximum F1 per dataset')
with pd.option_context('display.max_rows', None):
    display(oracle_f1_table.round(4))


[Oracle 01/32] apple
[Oracle 02/32] bank
[Oracle 03/32] bee_waggle_6
[Oracle 04/32] bitcoin
[Oracle 05/32] brent_spot
[Oracle 06/32] businv
[Oracle 07/32] children_per_woman
[Oracle 08/32] co2_canada
[Oracle 09/32] construction
[Oracle 10/32] gdp_argentina
[Oracle 11/32] gdp_iran
[Oracle 12/32] gdp_japan
[Oracle 13/32] global_co2
[Oracle 14/32] homeruns
[Oracle 15/32] iceland_tourism
[Oracle 16/32] jfk_passengers
[Oracle 17/32] lga_passengers
[Oracle 18/32] measles
[Oracle 19/32] nile
[Oracle 20/32] occupancy
[Oracle 21/32] ozone
[Oracle 22/32] ratner_stock
[Oracle 23/32] robocalls
[Oracle 24/32] run_log
[Oracle 25/32] scanline_126007
[Oracle 26/32] scanline_42049
[Oracle 27/32] seatbelts
[Oracle 28/32] shanghai_license
[Oracle 29/32] unemployment_nl
[Oracle 30/32] us_population
[Oracle 31/32] usd_isk
[Oracle 32/32] well_log
Overall Oracle results


,data type,Oracle F1,N datasets
0,multivariate,0.6074,4
1,univariate,0.7452,28


Oracle results excluding gdp_iran, gdp_japan, ozone, and robocalls


,data type,Oracle F1,N datasets
0,multivariate,0.6074,4
1,univariate,0.7028,24


Oracle-F1: maximum F1 per dataset


,dataset,data type,B,rho,q,Oracle F1
0,apple,multivariate,150,0.0500,0.990,0.6154
1,bank,univariate,25,0.0050,0.990,1.0000
2,bee_waggle_6,multivariate,25,0.0182,0.990,0.9286
3,bitcoin,univariate,25,0.2000,0.950,0.5063
4,brent_spot,univariate,150,0.0500,0.950,0.3607
5,businv,univariate,25,0.0182,0.900,0.5882
6,children_per_woman,univariate,25,0.0182,0.900,0.5075
7,co2_canada,univariate,25,0.0182,0.900,0.3610
8,construction,univariate,150,0.2000,0.900,0.8889
9,gdp_argentina,univariate,25,0.0182,0.990,0.8889
